# Welcome Back to Applying What You've Learned!

You have now completed two lessons: setting up project configurations and adding features with automatic skill selection. In both lessons, everything worked smoothly on the first attempt. However, real development rarely proceeds without mistakes, and that is perfectly fine. In fact, making mistakes and recovering from them is an essential part of learning and building software.

In this third lesson, we will explore how to recover from errors when building with Claude Code. We will intentionally make a mistake while adding validation to our Todo API, recognize the problem through test failures, and then work with Claude to correct the implementation. This practice will build your confidence to experiment freely, knowing that you can always fix issues when they arise.

---

## Understanding Conversation State

Before we practice error recovery, let us understand an important concept: **Claude Code maintains a complete history of your conversation**, including every file change, command execution, and tool use.

You can see your current session information using the `/status` command, which shows details like:

* Session ID
* Current working directory
* Configuration settings

When you are working with Claude and need to undo changes, you have several options:

* Ask Claude to **revert specific changes**.
* Use version control tools like **git**.
* **Restart** from a known good state.

> 💡 **Key Takeaway:** Claude can help you recover from mistakes by creating new corrected versions of files or by helping you navigate your version control history.

---

## Setting Up the Validation Task

We are going to add validation for the `priority` field in our Todo API. Currently, todos can have a priority of `low`, `medium`, or `high`, with `medium` as the default. However, there is no validation preventing invalid priority values from reaching our database. Let us add a validation middleware to solve this:

```text
> Add validation middleware for the priority field

```

This request is clear and focused. We want to validate the `priority` field before it reaches the route handler. Validation middleware is a common pattern in Express applications, sitting between the route and the handler to check input data.

Claude begins by reading the existing validation file to understand the current structure.

### Examining Current Validators

Claude first inspects what validators already exist:

```text
● Read(src/middleware/validate.js)
  ⎿ Current validators: validateTodoInput, validateEmail

```

The middleware file `src/middleware/validate.js` contains two validators: one for general todo input (`validateTodoInput`) and one for email addresses (`validateEmail`). Our new priority validator will follow the same pattern as these existing functions. This consistency is important because it makes the codebase easier to understand and maintain.

Now Claude is ready to add the new validator, but here is where we will **intentionally introduce a subtle mistake** that will help us learn about error recovery.

---

## Adding the Validation Middleware (With a Mistake)

Claude adds the `validatePriority` validation function to the middleware file:

```javascript
const validatePriority = (req, res, next) => {
  const { priority } = req.body;

  // Priority is REQUIRED (MISTAKE - should be optional!)
  if (!priority) {
    return res.status(400).json({ 
      error: 'Priority is required' 
    });
  }

  if (!['low', 'medium', 'high'].includes(priority)) {
    return res.status(400).json({ 
      error: 'Priority must be: low, medium, or high' 
    });
  }

  next();
};

```

This middleware contains a critical mistake: **it requires the priority field to be present.** The first check rejects any request where priority is missing. However, based on our requirements from the previous lesson, priority should be optional and default to `medium` when not specified. This validation is too strict and will break existing functionality.

The mistake is subtle because the code looks reasonable at first glance. The validation logic itself is correct (checking for valid values), but the requirement check (making it mandatory) contradicts our feature specification.

### Applying the Middleware to Routes

After creating the validator, Claude integrates it into the todo routes:

```javascript
const { validatePriority } = require('../middleware/validate');

router.post('/', authenticateToken, validatePriority, async (req, res) => {
  // ... todo creation logic
});

```

The middleware is now active in the request pipeline. Every request to create a todo must pass through `validatePriority` before reaching the handler. This is standard Express middleware composition, and the code follows our conventions perfectly.

At this point, we might feel satisfied: we added a validator and applied it correctly. But let us see what happens when we run the tests.

---

## Discovering the Problem

Claude runs the test suite to verify the changes:

```text
● Bash(npm test)
  ⎿ FAIL tests/priority.test.js
    Todo Priority Feature
      ✓ creates todo with priority
      ✗ defaults to medium priority when not specified
      ✓ filters todos by priority
      ✓ rejects invalid priority values
    
    Error: Priority is required
    Tests: 3 passed, 1 failed

```

We have a test failure. Three tests pass, but one fails: **the test that verifies defaulting to medium priority when the field is not specified.** The error message is telling: `Priority is required`. This is the exact error our validator returns when priority is missing from the request body.

This is the moment where we recognize something is wrong. The test was passing before, and now it fails after our changes. The failed test reveals the mistake: our validator incorrectly requires priority to be present, when it should actually be optional.

---

## Recovering from the Mistake

Now comes the important part: recovering from the mistake. We can simply ask Claude to fix the problem:

```text
> The validation is too strict - priority should be optional, not required. Please fix the validator.

```

This direct request tells Claude exactly what went wrong and what we need. Claude understands that the validation logic needs to change to make priority optional while still validating it when provided.

### Implementing the Correct Validation

Claude updates the validation to make priority optional:

```javascript
const validatePriority = (req, res, next) => {
  const { priority } = req.body;

  // Priority is OPTIONAL - only validate if provided
  if (priority && !['low', 'medium', 'high'].includes(priority)) {
    return res.status(400).json({ 
      error: 'Priority must be: low, medium, or high' 
    });
  }

  next();
};

```

The corrected version has one crucial change: **we removed the check that rejects missing priorities.** Now the validator only runs if priority is actually present in the request body (the `priority &&` condition). If priority is missing, the middleware calls `next()` immediately, allowing the request to proceed. The default value of `medium` will be applied later in the route handler or model, as designed.

This is the proper implementation. The validator ensures that when priority is provided, it must be one of the valid values. When priority is absent, the validator does not interfere.

The routes file does not need to change because the issue was in the validator logic, not in how it was applied. The middleware integration remains the same.

### Running Tests Successfully

Now for the moment of truth. Let us run the tests again with the corrected validation:

```text
● Bash(npm test)
  ⎿ PASS tests/priority.test.js
    Todo Priority Feature
      ✓ creates todo with priority (82ms)
      ✓ defaults to medium priority when not specified (68ms)
      ✓ filters todos by priority (91ms)
      ✓ rejects invalid priority values (43ms)
    
    Tests: 4 passed, 4 total

```

All tests pass. The second test, which failed before, now passes because our validator correctly allows requests without a priority field. The fourth test still passes because the validator correctly rejects invalid priority values when they are provided. The validation works exactly as intended.

This success demonstrates the complete error recovery cycle: recognize the problem through test failures, clearly communicate what needs to change, let Claude implement the fix, and verify success with tests.

---

## Understanding the Recovery Process

Let us reflect on what we just practiced:

```text
● What you practiced:
  1. ✓ Made a mistake (validation too strict)
  2. ✓ Recognized the error (test failure)
  3. ✓ Communicated the fix needed clearly
  4. ✓ Let Claude correct the implementation
  5. ✓ Verified with tests
  
  Key lesson: Tests catch mistakes early, and Claude can quickly 
  fix issues when you clearly describe what went wrong.

```

Each step in this recovery process is important. Making mistakes is inevitable; what matters is having automated tests to catch them and a clear communication path with Claude to fix them. The tests provide immediate feedback about what is wrong, and clear instructions to Claude enable quick corrections.

The final point is crucial: **this workflow enables confident experimentation.** When you know that tests will catch mistakes and Claude can help fix them, you feel more comfortable trying new approaches and testing ideas. Mistakes become learning opportunities rather than setbacks.

---

## Connecting to the Full Learning Path

We have now completed three lessons that build upon each other. Let us see how everything connects:

```text
● Unit 1 - Customizing Your Environment:
  ✓ Created CLAUDE.md for project instructions
  ✓ Set up .claudeignore for focus
  ✓ Configured permissions for safety
  
  Unit 2 - Using Skills Automatically:
  ✓ Reused api-tester skill across projects
  ✓ Observed automatic skill selection
  ✓ Saw consistent test generation
  
  Unit 3 - Recovering from Mistakes:
  ✓ Used slash commands for session management
  ✓ Let tests catch errors early
  ✓ Communicated fixes clearly to Claude
  ✓ Iterated confidently with quick corrections
  
  This is the full Claude Code workflow:
  Configure → Create Skills → Apply → Iterate

```

This summary shows how the three units work together to form a complete workflow. In Unit 1, we configured our project with `CLAUDE.md`, `.claudeignore`, and permissions. In Unit 2, we added features and watched skills like `api-tester` activate automatically. In Unit 3, we learned to recover from mistakes during iteration using tests and clear communication.

The workflow is cyclical: we configure our environment, create reusable skills, apply them to real tasks, and iterate with confidence knowing we can catch and fix errors. Each component supports the others, creating a robust development process.

---

## Conclusion and Next Steps

In this lesson, we practiced error recovery by intentionally making a mistake while adding validation, recognizing it through test failures, and then working with Claude to implement the correct solution. This experience demonstrates that mistakes are not permanent setbacks but rather temporary detours in the development process that automated testing helps us catch early.

The combination of automatic testing, clear communication with Claude, and quick iteration creates an environment where we can experiment freely. When we know that tests will catch mistakes and Claude can help fix them quickly, we feel more comfortable trying new approaches and learning from failures. This psychological safety is just as important as the technical capabilities.

You have now completed the core learning path for Claude Code: **project configuration, skill utilization, and error recovery.** These three pillars form the foundation of effective development with Claude. Now it is time to take what you have learned and apply it through hands-on practice, where you will tackle real challenges and build your confidence in this powerful workflow!

## Recovering from Overly Strict Validation

Mistakes occur in real-world projects, and knowing how to fix them is just as important as writing code in the first place. In this exercise, you will practice the full cycle of introducing a problem, discovering it through tests, and recovering from it.

Your task is to ask Claude to add input validation for the description field in the todo creation endpoint. Here is the twist: you will intentionally request a validation that is too strict, and you will tell Claude upfront to fix the implementation if tests fail.

Ask Claude something like:

"Add input validation for the description field in the todo creation endpoint. Make the description field required. Then run the tests. If any tests fail, fix the validation code to match what the tests expect - don't change the tests."

You can choose either:

    Ask for a maximum length of 50 characters (when it should be 500).
    Ask Claude to make the description field required (when it should be optional).

The key is adding the instruction: "If tests fail, fix the validation code to match what the tests expect - don't change the tests."

This ensures Claude will implement your intentionally wrong requirement, run tests, see them fail, and then fix the implementation to match the correct requirements encoded in the tests. This demonstrates the proper error recovery workflow: tests define the requirements, and code should be changed to satisfy tests, not the other way around.

To solve this task successfully and demonstrate the proper error recovery workflow with Claude Code, follow these step-by-step actions:

### Step 1: Open Your Claude Code Session

Ensure you are in your project's root directory where your Express application and test suite are located. You can verify your environment configuration by typing:

```text
/status

```

*(This command confirms your current working directory and session state before you begin modifying files.)*

---

### Step 2: Submit the Intentionally Flawed Prompt

Copy and paste the exact instruction into your Claude Code terminal. This prompt combines an overly strict requirement with explicit recovery instructions:

```text
Add input validation for the description field in the todo creation endpoint. Make the description field required. Then run the tests. If any tests fail, fix the validation code to match what the tests expect - don't change the tests.

```

---

### Step 3: Observe Claude's Execution Pipeline

Watch how Claude processes your request through its automated tool execution loop:

1. **File Reading:** Claude will inspect your existing middleware (likely `src/middleware/validate.js`) and your routes file to understand how input validation is currently structured.
2. **Code Modification (The Mistake):** Claude will write the validation middleware to explicitly check for and require `description`, mirroring the intentional error pattern:
```javascript
// Claude's initial wrong implementation
if (!description) {
  return res.status(400).json({ error: 'Description is required' });
}

```


3. **Route Integration:** Claude will mount this new validator onto your POST route pipeline.

---

### Step 4: Analyze the Automatic Test Failure

Because your prompt instructs Claude to run the tests immediately, it will trigger the test suite via a shell execution command (e.g., `npm test`).

You will see a terminal readout showing a test failure similar to this:

```text
● Bash(npm test)
  ⎿ FAIL tests/todo.test.js
    ✗ should create a todo successfully when description is omitted
    
    Error: Expected status 201, got 400 (Description is required)

```

This failure provides immediate feedback that your new requirement contradicts the application's underlying technical specifications (where description is supposed to be optional).

---

### Step 5: Review the Automatic Autonomous Recovery

Because you appended the conditional instruction (*"If any tests fail, fix the validation code..."*), **you do not need to type a second prompt.** Claude will read the test output logs, recognize that making the field mandatory broke the integration parameters, and automatically modify the file to change `description` from required to optional:

```javascript
// Claude's automated recovery implementation
if (description && description.length > 500) { 
  // Evaluates description parameters safely without forcing it to exist
}

```

---

### Step 6: Verify Final Test Success

Claude will run the test suite one last time to confirm that its corrections successfully satisfy all assertions. Look for the clean pass status:

```text
● Bash(npm test)
  ⎿ PASS tests/todo.test.js
    ✓ should create a todo successfully when description is omitted
    ✓ should reject invalid descriptions

```

Once all tests pass, the error recovery cycle is complete. You have successfully let automated tests guard the integrity of your codebase while leveraging Claude to handle the iteration loop safely!

## Fixing Bugs in Completion Tracking

Now that you've successfully recovered from a validation issue, let's tackle something trickier: observing how bugs in more complex feature logic get caught and fixed through testing.

Your task is to add a new feature that tracks when to-dos are completed. Ask Claude to add a "completed date" field that automatically records the timestamp when a todo is marked as complete. Let Claude implement this feature naturally — bugs might sneak in, and that's exactly what we want to practice observing.

Once Claude has implemented the feature and runs the comprehensive test suite, watch carefully as the automated tests reveal issues. Claude will autonomously identify and fix problems such as:

    Type conversion errors when SQLite expects numbers instead of booleans
    Server configuration issues that prevent tests from running correctly
    Database binding errors with incompatible data types

Your role is to observe Claude's debugging process. Watch how Claude:

    Reads test failure messages to understand what went wrong
    Identifies the root cause of errors (e.g., "SQLite3 can only bind numbers, strings, bigints, buffers, and null")
    Implements fixes autonomously
    Re-runs tests to verify the fixes work

This exercise shows you how testing helps catch subtle bugs that are easy to miss — and how Claude can autonomously debug and fix issues by carefully reading test output and understanding error messages. This is a realistic development workflow where automated testing provides continuous feedback during feature implementation.

To complete this learning exercise and observe how Claude Code handles automatic debugging of more complex feature logic, you can follow the step-by-step guide below:

## Step 1: Send New Feature Prompt to Claude Code

Copy and paste the instruction below into your Claude Code terminal. This prompt asks Claude to add a completion tracking feature while instructing it to run tests and fix any bugs that arise autonomously:

```text
Add a "completed date" field to the todo model and API. This field should automatically record the current timestamp when a todo's completed status is set to true, and clear it (set to null) if it is changed back to false. After implementing the feature, run the test suite. If any database, type binding, or SQLite validation errors occur, debug and fix them autonomously.
```

---

## Step 2: Observe the Feature Implementation Process

Claude will start working autonomously in the following sequence:

1. **Modify Database Schema/Model:** Claude will read your model or database migration files to add a new column (for example, `completed_at` or `completedDate`).
2. **Update Controller/Route Logic:** Claude will modify the task update function. If the `completed` status is `true`, it will insert `new Date()`. If `false`, it will change the value to `null`.

---

## Step 3: Witness Bug Emergence Through Automated Testing

After the code is written, Claude will immediately run the test command (such as `npm test`). Since local test environments often use different databases (like **SQLite**), differences in data type handling usually trigger errors.

Watch your terminal - most likely you will see test failure messages like this:

```text
● Bash(npm test)
  ⎿ FAIL tests/completion.test.js
    ✗ should automatically record timestamp when completed is true

    Error: table todos has no column named completed_at
    OR
    TypeError: SQLite3 can only bind numbers, strings, bigints, buffers, and null
```

---

## Step 4: Observe Claude's Autonomous Debugging Process

This is where the core of this exercise lies. Since you provided handling instructions at the beginning, **don't type anything in the terminal**. Observe how Claude analyzes the failure autonomously:

1. **Reading the Error Log:** Claude reads the error text and understands that SQLite doesn't support raw date objects (`Date object`) or the database column hasn't been properly migrated in the test environment.
2. **Identifying the Root Cause:** Claude concludes that it needs to convert the JavaScript `Date` object to an ISO string (`new Date().toISOString()`) or a numeric time format before sending it to the SQLite query.
3. **Applying Code Fix:** Claude opens the problematic file automatically and fixes the data type conversion or database binding configuration.

---

## Step 5: Verify Final Results

After making internal fixes, Claude will automatically trigger the test command again to ensure its solution works. You will see the test status turn green:

```text
● Bash(npm test)
  ⎿ PASS tests/completion.test.js
    ✓ should automatically record timestamp when completed is true
    ✓ should clear timestamp when completed is set back to false

    Tests: All tests passed!
```

---

## Exercise Conclusion

Through this observation exercise, you can directly see how **Automated Testing** acts as a powerful safety net. Subtle bugs like data type mismatches between application code and the database are immediately detected before reaching production, and Claude can use them as accurate guides to fix systems quickly!

---

## Recovery from Mistakes

When working with Claude, there are several key pieces of information you should be aware of that can help you recover from errors:

* Session ID
* Current working directory
* Configuration settings

When you are working with Claude and need to undo changes, you have several options:

* Ask Claude to **revert specific changes**.
* Use version control tools like **git**.
* **Restart** from a known good state.

> 💡 **Key Takeaway:** Claude can help you recover from mistakes by creating new corrected versions of files or by helping you navigate your version control history.

---

## Setting Up the Validation Task

We are going to add validation for the `priority` field in our Todo API. Currently, todos can have a priority of `low`, `medium`, or `high`, with `medium` as the default. However, there is no validation preventing invalid priority values from reaching our database. Let us add a validation middleware to solve this:

```text
> Add validation middleware for the priority field
```

This request is clear and focused. We want to validate the `priority` field before it reaches the route handler. Validation middleware is a common pattern in Express applications, sitting between the route and the handler to check input data.

Claude begins by reading the existing validation file to understand the current structure.

### Examining Current Validators

Claude first inspects what validators already exist:

```text
● Read(src/middleware/validate.js)
  ⎿ Current validators: validateTodoInput, validateEmail
```

The middleware file `src/middleware/validate.js` contains two validators: one for general todo input (`validateTodoInput`) and one for email addresses (`validateEmail`). Our new priority validator will follow the same pattern as these existing functions. This consistency is important because it makes the codebase easier to understand and maintain.

Now Claude is ready to add the new validator, but here is where we will **intentionally introduce a subtle mistake** that will help us learn about error recovery.

---

## Adding the Validation Middleware (With a Mistake)

Claude adds the `validatePriority` validation function to the middleware file:

```javascript
const validatePriority = (req, res, next) => {
  const { priority } = req.body;

  // Priority is REQUIRED (MISTAKE - should be optional!)
  if (!priority) {
    return res.status(400).json({ 
      error: 'Priority is required' 
    });
  }

  if (!['low', 'medium', 'high'].includes(priority)) {
    return res.status(400).json({ 
      error: 'Priority must be: low, medium, or high' 
    });
  }

  next();
};
```